# 02 — Cleaning

## Cohort Decisions

Three filters define the analytical population:

1. **Delivered orders only** — drivers of dissatisfaction differ categorically 
   for non-delivered orders. Including them dilutes the signal.

2. **Orders with reviews only** — required by the question structure. 
   Acknowledged limitation: introduces selection bias toward customers 
   who choose to review.

3. **No timing filter on review date** — but flag review-to-delivery time 
   gap as a potential analytical limitation in findings.

In [3]:
import pandas as pd

# Load tables
orders = pd.read_csv('../data/olist_orders_dataset.csv')
order_items = pd.read_csv('../data/olist_order_items_dataset.csv')
reviews = pd.read_csv('../data/olist_order_reviews_dataset.csv')
products = pd.read_csv('../data/olist_products_dataset.csv')
sellers = pd.read_csv('../data/olist_sellers_dataset.csv')
category_translation = pd.read_csv('../data/product_category_name_translation.csv')

In [4]:
# ---- Filter 1: Delivered orders only ----
print(f"Total orders: {len(orders):,}")
delivered = orders[orders['order_status'] == 'delivered'].copy()
print(f"After delivered filter: {len(delivered):,}")

# ---- Filter 2: Orders with reviews ----
reviewed_order_ids = reviews['order_id'].unique()
analysis_orders = delivered[delivered['order_id'].isin(reviewed_order_ids)].copy()
print(f"After review filter: {len(analysis_orders):,}")

# Final analytical population size
print(f"\nAnalytical population: {len(analysis_orders):,} orders")
print(f"% retained from original: {len(analysis_orders)/len(orders)*100:.1f}%")

Total orders: 99,441
After delivered filter: 96,478
After review filter: 95,832

Analytical population: 95,832 orders
% retained from original: 96.4%


## Type Conversions

| Column | Current | Target | Why |
|---|---|---|---|
| order_purchase_timestamp | object | datetime64 | Time arithmetic |
| order_approved_at | object | datetime64 | Time arithmetic |
| order_delivered_carrier_date | object | datetime64 | Time arithmetic |
| order_delivered_customer_date | object | datetime64 | Time arithmetic |
| order_estimated_delivery_date | object | datetime64 | Time arithmetic |
| order_status | object | category | Memory + semantics |

In [5]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    analysis_orders[col] = pd.to_datetime(analysis_orders[col])

analysis_orders['order_status'] = analysis_orders['order_status'].astype('category')
print(analysis_orders.dtypes)

order_id                                    str
customer_id                                 str
order_status                           category
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [6]:
analysis_orders.info()

<class 'pandas.DataFrame'>
Index: 95832 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       95832 non-null  str           
 1   customer_id                    95832 non-null  str           
 2   order_status                   95832 non-null  category      
 3   order_purchase_timestamp       95832 non-null  datetime64[us]
 4   order_approved_at              95818 non-null  datetime64[us]
 5   order_delivered_carrier_date   95830 non-null  datetime64[us]
 6   order_delivered_customer_date  95824 non-null  datetime64[us]
 7   order_estimated_delivery_date  95832 non-null  datetime64[us]
dtypes: category(1), datetime64[us](5), str(2)
memory usage: 11.8 MB


In [7]:
print("Missing values in analysis_orders:")
print(analysis_orders.isna().sum())
print(f"\nTotal rows: {len(analysis_orders):,}")

Missing values in analysis_orders:
order_id                          0
customer_id                       0
order_status                      0
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
dtype: int64

Total rows: 95,832


## Missing Value Strategy

The cleaned cohort has minimal missing data (<0.03% per column). 
Affected columns are all delivery-related dates that we'll need for 
delivery time analysis. Given the trivial volume, we drop these rows 
rather than impute.

In [8]:
# Track size before drop for audit
before = len(analysis_orders)

# Drop rows missing any delivery date
analysis_orders = analysis_orders.dropna(subset=[
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date'
])

after = len(analysis_orders)
print(f"Dropped {before - after} rows with missing delivery dates")
print(f"Final cohort: {after:,} orders")

Dropped 23 rows with missing delivery dates
Final cohort: 95,809 orders


In [9]:
items_per_order = order_items.groupby('order_id').size()
print(f"Total orders with items: {len(items_per_order):,}")
print(f"Single-item orders: {(items_per_order == 1).sum():,} ({(items_per_order == 1).mean()*100:.1f}%)")
print(f"Multi-item orders: {(items_per_order > 1).sum():,} ({(items_per_order > 1).mean()*100:.1f}%)")
print(f"\nDistribution:")
print(items_per_order.value_counts().sort_index().head(10))

Total orders with items: 98,666
Single-item orders: 88,863 (90.1%)
Multi-item orders: 9,803 (9.9%)

Distribution:
1     88863
2      7516
3      1322
4       505
5       204
6       198
7        22
8         8
9         3
10        8
Name: count, dtype: int64


## Building the Analysis Dataset

Grain: one row per order. For orders with multiple items, we use the first 
item as representative for product/seller attributes and capture multi-item 
complexity through aggregate features (item_count, unique_sellers).

In [10]:
# Aggregate order_items to one row per order

order_features = order_items.groupby('order_id').agg(
    item_count=('order_item_id', 'count'),
    unique_sellers=('seller_id', 'nunique'),
    unique_products=('product_id', 'nunique'),
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
    first_seller_id=('seller_id', 'first'),
    first_product_id=('product_id', 'first'),
).reset_index()

print(f"order_features shape: {order_features.shape}")
print(f"Expected: ~{len(order_items['order_id'].unique()):,} rows (one per order)")
print(order_features.head())

order_features shape: (98666, 8)
Expected: ~98,666 rows (one per order)
                           order_id  item_count  unique_sellers  \
0  00010242fe8c5a6d1ba2dd792cb16214           1               1   
1  00018f77f2f0320c557190d7a144bdd3           1               1   
2  000229ec398224ef6ca0657da4fc703e           1               1   
3  00024acbcdf0a6daa1e931b038114c75           1               1   
4  00042b26cf59d7ce69dfabb4e55b4fd9           1               1   

   unique_products  total_price  total_freight  \
0                1        58.90          13.29   
1                1       239.90          19.93   
2                1       199.00          17.87   
3                1        12.99          12.79   
4                1       199.90          18.14   

                    first_seller_id                  first_product_id  
0  48436dade18ac8b2bce089ec2a041202  4244733e06e7ecb4970a6e2683c13e61  
1  dd7ddc04e1b6c2c614352b383efe2d36  e5f2d52b802189ee658865ca93d83a8f  
2  5b510

## The Joins

We join in this order:
1. `analysis_orders` (delivered + reviewed cohort) <- spine
2. + `order_features` (aggregated item info)
3. + `reviews` (review_score — dependent variable)
4. + `products` (product attributes via first_product_id)
5. + `sellers` (seller attributes via first_seller_id)

Each join is a LEFT join from the spine to preserve all 95,809 cohort orders.

In [19]:
# Start with cohort orders as spine
analysis_df = analysis_orders.copy()
print(f"Start: {len(analysis_df):,} rows")

# Join 1: order features
analysis_df = analysis_df.merge(order_features, on='order_id', how='left')
print(f"After order_features: {len(analysis_df):,} rows")

# Join 2: reviews — keep only review_score and review timestamps
review_cols = reviews[['order_id', 'review_score', 'review_creation_date', 'review_answer_timestamp']]
analysis_df = analysis_df.merge(review_cols, on='order_id', how='left')
print(f"After reviews: {len(analysis_df):,} rows")

Start: 95,809 rows
After order_features: 95,809 rows
After reviews: 96,338 rows


were using left join, which means that i should have same number of rows throughout as i join tables, lets find out why

In [20]:
# How many reviews per order?
reviews_per_order = reviews.groupby('order_id').size()
print(f"Max reviews per order: {reviews_per_order.max()}")
print(f"Orders with multiple reviews: {(reviews_per_order > 1).sum()}")
print(f"\nDistribution:")
print(reviews_per_order.value_counts().sort_index())

Max reviews per order: 3
Orders with multiple reviews: 547

Distribution:
1    98126
2      543
3        4
Name: count, dtype: int64


In [21]:
# Reset analysis_df to before the buggy reviews join
analysis_df = analysis_orders.copy()
analysis_df = analysis_df.merge(order_features, on='order_id', how='left')

# Aggregate reviews to one row per order (lowest score wins)
reviews_one_per_order = (
    reviews
    .sort_values('review_score', ascending=True) # lowest first
    .drop_duplicates(subset='order_id', keep='first') # keep first occurrence = lowest
     [['order_id', 'review_score', 'review_creation_date', 'review_answer_timestamp']]
)

# Verify
print(f"reviews_one_per_order shape: {reviews_one_per_order.shape}")
print(f"Unique order_ids: {reviews_one_per_order['order_id'].nunique():,}")

# Now join cleanly
analysis_df = analysis_df.merge(reviews_one_per_order, on='order_id', how='left')
print(f"After reviews join: {len(analysis_df):,} rows")

reviews_one_per_order shape: (98673, 4)
Unique order_ids: 98,673
After reviews join: 95,809 rows


In [22]:
# Join 4: Products
analysis_df = analysis_df.merge(
    products[['product_id', 'product_category_name', 'product_weight_g', 
              'product_length_cm', 'product_height_cm', 'product_width_cm',
              'product_photos_qty']],
    left_on='first_product_id',
    right_on='product_id',
    how='left'
)
print(f"After products: {len(analysis_df):,} rows")

# Join 5: Sellers
analysis_df = analysis_df.merge(
    sellers[['seller_id', 'seller_state']],
    left_on='first_seller_id',
    right_on='seller_id',
    how='left'
)
print(f"After sellers: {len(analysis_df):,} rows")

# Join 6: Category translation (for readability)
analysis_df = analysis_df.merge(
    category_translation,
    on='product_category_name',
    how='left'
)
print(f"After category translation: {len(analysis_df):,} rows")

print(f"\nFinal analysis_df shape: {analysis_df.shape}")
print(f"Columns: {analysis_df.columns.tolist()}")

After products: 95,809 rows
After sellers: 95,809 rows
After category translation: 95,809 rows

Final analysis_df shape: (95809, 28)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'item_count', 'unique_sellers', 'unique_products', 'total_price', 'total_freight', 'first_seller_id', 'first_product_id', 'review_score', 'review_creation_date', 'review_answer_timestamp', 'product_id', 'product_category_name', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_photos_qty', 'seller_id', 'seller_state', 'product_category_name_english']


In [23]:
analysis_df.to_csv('../data/analysis_df.csv', index=False)
print(f"Saved analysis_df.csv with {len(analysis_df):,} rows")

Saved analysis_df.csv with 95,809 rows
